# UC10 — Optimización Energética de la Planta

**Objetivo:** Recomendar la configuración óptima de equipos para minimizar coste energético adaptándose a tarifas eléctricas variables.

**Tecnologías:** ML.CLASSIFICATION · Streamlit

**Tiempo estimado:** 14 minutos

## Paso 1 — Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS OPTIMIZACION_ENERGIA_DB;

In [ ]:
USE DATABASE OPTIMIZACION_ENERGIA_DB;

In [ ]:
USE SCHEMA PUBLIC;

In [ ]:
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;

In [ ]:
USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar Datos de Operación y Tarifas

8.760 registros (365 días × 24 horas) con consumo energético y tarifas eléctricas.

In [ ]:
CREATE OR REPLACE TABLE operacion_historica AS
SELECT
    DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP()) AS timestamp,
    ROUND(100 + 100 * SIN(SEQ4() * 3.14159 / 12) + UNIFORM(-20, 30, RANDOM()), 0) AS demanda_m3h,
    CASE WHEN HOUR(DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP())) BETWEEN 10 AND 14 OR HOUR(DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP())) BETWEEN 18 AND 22 THEN 'Punta'
         WHEN HOUR(DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP())) BETWEEN 8 AND 10 OR HOUR(DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP())) BETWEEN 14 AND 18 THEN 'Llano'
         ELSE 'Valle' END AS periodo,
    CASE WHEN HOUR(DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP())) BETWEEN 10 AND 14 OR HOUR(DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP())) BETWEEN 18 AND 22 THEN ROUND(0.15 + UNIFORM(0, 7, RANDOM()) * 0.01, 3)
         WHEN HOUR(DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP())) BETWEEN 8 AND 10 OR HOUR(DATEADD('hour', -SEQ4(), CURRENT_TIMESTAMP())) BETWEEN 14 AND 18 THEN ROUND(0.08 + UNIFORM(0, 4, RANDOM()) * 0.01, 3)
         ELSE ROUND(0.04 + UNIFORM(0, 3, RANDOM()) * 0.01, 3) END AS precio_kwh,
    (1 + UNIFORM(0, 4, RANDOM()))::INT AS num_bombas_activas,
    ROUND(10 + UNIFORM(0, 20, RANDOM()), 1) AS temp_ambiente_c,
    ROUND((100 + 100 * SIN(SEQ4() * 3.14159 / 12)) * (0.8 + UNIFORM(0, 40, RANDOM()) * 0.01), 0) AS consumo_kwh,
    CASE WHEN UNIFORM(0, 100, RANDOM()) < 95 THEN TRUE ELSE FALSE END AS calidad_salida_ok
FROM TABLE(GENERATOR(ROWCOUNT => 8760));

In [ ]:
SELECT periodo, COUNT(*) AS horas, ROUND(AVG(precio_kwh),3) AS precio_medio, ROUND(AVG(consumo_kwh),0) AS consumo_medio FROM operacion_historica GROUP BY 1;

## Paso 3 — Crear Bandas de Eficiencia Energética

Clasificamos cada hora en bandas según la eficiencia (coste por m³ producido).

In [ ]:
CREATE OR REPLACE TABLE operacion_bandas AS
SELECT *,
    ROUND(consumo_kwh * precio_kwh / NULLIF(demanda_m3h, 0), 4) AS coste_por_m3,
    CASE
        WHEN (consumo_kwh * precio_kwh / NULLIF(demanda_m3h, 0)) < 0.05 AND calidad_salida_ok THEN 'Optima'
        WHEN (consumo_kwh * precio_kwh / NULLIF(demanda_m3h, 0)) < 0.08 AND calidad_salida_ok THEN 'Aceptable'
        WHEN (consumo_kwh * precio_kwh / NULLIF(demanda_m3h, 0)) < 0.12 THEN 'Mejorable'
        ELSE 'Ineficiente'
    END AS banda_config
FROM operacion_historica;

In [ ]:
SELECT banda_config, COUNT(*) AS horas, ROUND(AVG(coste_por_m3), 4) AS coste_medio FROM operacion_bandas GROUP BY 1 ORDER BY 3;

## Paso 4 — Entrenar Modelo de Optimización

In [ ]:
CREATE OR REPLACE TABLE features_energia AS
SELECT
    demanda_m3h::FLOAT, precio_kwh::FLOAT,
    CASE WHEN periodo = 'Punta' THEN 1 ELSE 0 END::FLOAT AS periodo_punta,
    CASE WHEN periodo = 'Valle' THEN 1 ELSE 0 END::FLOAT AS periodo_valle,
    temp_ambiente_c::FLOAT, num_bombas_activas::FLOAT,
    HOUR(timestamp)::FLOAT AS hora_del_dia,
    banda_config
FROM operacion_bandas;

In [ ]:
CREATE OR REPLACE TABLE train_energia AS SELECT * FROM features_energia SAMPLE(80);

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION optimizador_energia(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'train_energia'),
    TARGET_COLNAME => 'banda_config',
    CONFIG_OBJECT => {'evaluate': TRUE}
);

## Paso 5 — Calcular Ahorro Potencial

In [ ]:
SELECT
    periodo,
    ROUND(SUM(consumo_kwh * precio_kwh), 0) AS coste_actual,
    ROUND(SUM(consumo_kwh * precio_kwh * 0.85), 0) AS coste_optimizado,
    ROUND(SUM(consumo_kwh * precio_kwh) - SUM(consumo_kwh * precio_kwh * 0.85), 0) AS ahorro_estimado,
    ROUND(SUM(consumo_kwh) * 0.15 * 0.00025, 1) AS co2_evitado_toneladas
FROM operacion_historica
GROUP BY periodo;

## Paso 6 — Dashboard Energético

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()

st.title("⚡ Optimización Energética de Planta")

col1, col2, col3 = st.columns(3)
stats = session.sql("SELECT ROUND(SUM(consumo_kwh * precio_kwh),0) AS coste_total, ROUND(SUM(consumo_kwh * precio_kwh * 0.15),0) AS ahorro, ROUND(SUM(consumo_kwh) * 0.15 * 0.00025, 1) AS co2 FROM operacion_historica").collect()[0]
col1.metric("Coste Energético Anual", f"{stats['COSTE_TOTAL']:,}€")
col2.metric("Ahorro Potencial", f"{stats['AHORRO']:,}€")
col3.metric("CO₂ Evitable", f"{stats['CO2']} t")

st.subheader("Coste por Periodo Tarifario")
df = session.sql("SELECT periodo, ROUND(SUM(consumo_kwh * precio_kwh),0) AS coste FROM operacion_historica GROUP BY 1 ORDER BY 2 DESC").to_pandas()
st.bar_chart(df.set_index("PERIODO"))

## Paso 7 — Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS OPTIMIZACION_ENERGIA_DB;